# Experiment 1 — GE-MolSG vs MolSG (DUDE-Z retrieval)

Per-target retrieval performance (EF1%, BEDROC) for two methods:

- **GE-MolSG** — this package's WKS → hard k-NN BoF (`knn_histogram`) over a geo
  codebook, chi-squared-kernel retrieval.
- **MolSG** — FEM Laplace–Beltrami → WKS → hard-assignment BoF baseline
  (`molsg` package), cosine retrieval.

Results are generated **iteratively per target** and cached to disk, so a rerun
skips completed targets. After all targets finish, per-target means are
summarised, two scatter/line plots (BEDROC and EF1%) are produced, and a
Wilcoxon signed-rank test (focused on GE-MolSG) is run.


### Data layout (Zenodo-style)
```
<DATA_ROOT>/DUDE-Z/<TARGET>.tar.gz  ->  <TARGET>/ESP_Npy/{ligand,decoy}_<ID>.npy
```
### Codebooks
```
experiments/codebooks/ge_molsg_cb.npy    # GE-MolSG geo codebook
experiments/codebooks/molsg.npy          # MolSG baseline codebook
```
### Queries
```
<QUERIES_DIR>/<target>.csv  with a 'queries' column of active IDs
```


## 1 · Configuration
Edit these paths.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import logging
import sys
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd

# ── Experiment paths (EDIT THESE) ────────────────────────────────────────────
TARGETS = [
    "AA2AR", "ABL1", "ACES", "ADA", "ADRB2", "AMPC", "ANDR", "CSF1R",
    "CXCR4", "DEF", "DRD4", "EGFR", "FA10", "FA7", "FABP4", "FGFR1",
    "FKB1A", "GLCM", "HDAC8", "HIVPR", "HMDH", "HS90A", "ITAL", "KIT",
    "KITH", "LCK", "MAPK2", "MK01", "MT1", "NRAM", "PARP1", "PLK1",
    "PPARA", "PTN1", "PUR2", "RENI", "ROCK1", "SRC", "THRB", "TRY1",
    "TRYB1", "UROK", "XIAP",
]  # comment out any targets you don't want to run
# Data: per-target archives <TARGET>.tar.gz are hosted in one Zenodo record.
# Set ZENODO_BASE_URL and each listed target is downloaded + extracted on demand
# (only the targets in TARGETS are fetched). Each <TARGET>.tar.gz extracts to
# <TARGET>/{ESP_Npy, ESP_Npy_MMFF94, PDB_Files}/.
ZENODO_BASE_URL = "https://zenodo.org/records/20547837/files"   # e.g. "https://zenodo.org/records/XXXXXXX/files"
DATA_ROOT    = Path("dude_z_data")   # local cache for downloaded/extracted targets
QUERIES_DIR  = Path("queries")
OUT_DIR      = Path("experiments_out/exp1_gemolsg_vs_molsg")
CODEBOOK_DIR = Path("codebooks")

GE_CODEBOOK    = CODEBOOK_DIR / "ge_molsg_cb.npy"
MOLSG_CODEBOOK = CODEBOOK_DIR / "molsg.npy"

METHODS = ["GE-MolSG", "MolSG"]
METRICS = ["EF1%", "BEDROC"]

# ── Descriptor params ────────────────────────────────────────────────────────
K        = 100      # LBO eigenvalues (n_components)
EVALS    = 100      # WKS evals
VAR      = 15       # WKS variance
KNN      = 100      # graph kNN
EW       = 0.3      # electrostatic weight
LAP_NORM = "normalized"
BOF_KNN  = 3

FORCE   = False     # recompute even if a target's CSV is cached



In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT = OUT_DIR / "results"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(OUT_DIR / "exp1_gemolsg_vs_molsg.log")],
    force=True,
)
log = logging.getLogger("exp1")
log.info("Experiment 1 — GE-MolSG vs MolSG")
log.info("Targets: %s", ", ".join(TARGETS))


## 2 · Data / queries helpers

In [ ]:
import subprocess
import urllib.request


def fetch_target(target, data_root):
    """Ensure <data_root>/<TARGET>/ exists, downloading from Zenodo if needed.

    Looks for an already-extracted ``<TARGET>/ESP_Npy`` first. If absent and
    ``ZENODO_BASE_URL`` is set, downloads ``<ZENODO_BASE_URL>/<TARGET>.tar.gz``
    and extracts it under ``data_root``. Each archive extracts to
    ``<TARGET>/{ESP_Npy, ESP_Npy_MMFF94, PDB_Files}/``.

    Returns the ``<TARGET>/`` directory.
    """
    data_root = Path(data_root)
    for cand in (data_root / target, data_root / target.upper(),
                 data_root / "DUDE-Z" / target):
        if (cand / "ESP_Npy").is_dir():
            return cand

    local_tar = None
    for tar in (data_root / f"{target}.tar.gz",
                data_root / "DUDE-Z" / f"{target}.tar.gz"):
        if tar.exists():
            local_tar = tar
            break

    if local_tar is None:
        if not ZENODO_BASE_URL:
            raise FileNotFoundError(
                f"No local data for '{target}' and ZENODO_BASE_URL is not set.")
        data_root.mkdir(parents=True, exist_ok=True)
        local_tar = data_root / f"{target}.tar.gz"
        url = f"{ZENODO_BASE_URL.rstrip('/')}/{target}.tar.gz"
        log.info("Downloading %s", url)
        try:
            subprocess.run(["curl", "-fSL", "-o", str(local_tar), url], check=True)
        except (FileNotFoundError, subprocess.CalledProcessError):
            urllib.request.urlretrieve(url, local_tar)

    log.info("Extracting %s", local_tar)
    with tarfile.open(local_tar, "r:gz") as tf:
        tf.extractall(data_root)

    for cand in (data_root / target, data_root / target.upper()):
        if (cand / "ESP_Npy").is_dir():
            return cand
    hits = list(data_root.rglob(f"{target}/ESP_Npy")) or list(data_root.rglob("ESP_Npy"))
    if hits:
        return hits[0].parent
    raise FileNotFoundError(f"Extracted '{target}' but found no ESP_Npy under {data_root}")


def resolve_target_dir(target, data_root, work=None):
    """Return the ESP_Npy directory for a target."""
    target_root = fetch_target(target, data_root)
    esp_dir = target_root / "ESP_Npy"
    if not esp_dir.is_dir():
        raise FileNotFoundError(f"No ESP_Npy for '{target}' under {target_root}")
    return esp_dir

def resolve_queries_csv(queries_dir, target):
    for name in (f"{target}.csv", f"{target.lower()}.csv", f"{target.upper()}.csv"):
        if (queries_dir / name).exists():
            return queries_dir / name
    raise FileNotFoundError(f"No queries CSV for '{target}' under {queries_dir}")

def load_query_ids(csv_path):
    """Read query IDs from the CSV, using whichever query column is present."""
    df = pd.read_csv(csv_path)
    col = next((c for c in ("queries", "query") if c in df.columns), None)
    if col is None:
        # fall back to the single column if unambiguous
        if df.shape[1] == 1:
            col = df.columns[0]
        else:
            raise ValueError(f"{csv_path}: no 'query'/'queries' column found")
    return [str(q).strip() for q in df[col].dropna()]

def list_surface_fns(surf_dir):
    return sorted([f for f in os.listdir(surf_dir) if f.endswith(".npy")], reverse=True)

## 3 · Retrieval metrics
Chi-squared kernel for GE-MolSG; cosine for MolSG. EF1% and BEDROC (α=20).

In [ ]:
from sklearn.metrics.pairwise import chi2_kernel, cosine_similarity
from rdkit.ML.Scoring import Scoring

SIM_KIND = {"GE-MolSG": "chi2", "MolSG": "cosine"}

def sim_matrix(vectors, kind):
    V = np.asarray(vectors, dtype=np.float64)
    return chi2_kernel(V) if kind == "chi2" else cosine_similarity(V)

def query_metrics(sim_row, labels, i):
    keep = np.arange(len(labels)) != i
    sim = sim_row[keep].astype(np.float64)
    lab = labels[keep].astype(np.float64)
    order = np.argsort(sim)[::-1]
    scores = np.column_stack([sim[order], lab[order]])
    ef = Scoring.CalcEnrichment(scores, 1, [0.01])
    bedroc = Scoring.CalcBEDROC(scores, 1, 20)
    return {"EF1%": float(ef[0]), "BEDROC": float(bedroc)}

def retrieve(vectors, fns, query_ids, sim_kind):
    labels = np.asarray([1 if f[0] == "l" else 0 for f in fns])
    stems = [f[:-6] for f in fns]
    log.info(stems)
    S = sim_matrix(vectors, sim_kind)
    rows = []
    for qid in query_ids:
        hits = [j for j, s in enumerate(stems) if s == qid or s.endswith(qid)]
        if not hits:
            log.warning("  query '%s' not found; skipping", qid)
            continue
        i = hits[0]
        m = query_metrics(S[i], labels, i)
        m["RefMol"] = stems[i]
        rows.append(m)
    return pd.DataFrame(rows, columns=["RefMol", "EF1%", "BEDROC"])

## 4 · Encoders
Encoders for each method.

In [ ]:
import ge_molsg as gm

def encode_gemolsg(surf_dir, fns):
    """GE-MolSG WKS -> hard k-NN BoF over the geo codebook."""
    surfaces = [gm.load_surface_npy(str(surf_dir / f), name=f[:-4]) for f in fns]
    vecs = []
    codebook = np.load(GE_CODEBOOK, allow_pickle=True)
    for surf in surfaces:
        pts = surf.augmented_points(elec_weight=EW)
        W = gm.compute_affinity(
            pts, n_neighbors=KNN, backend="ckdtree",
            adaptive_bw=True, square_distances=False
        )
        L = gm.graph_laplacian(W, laplacian_type=LAP_NORM)
        ev, evec = gm.compute_eigenpairs(
            L, n_components=K, drop_first=True,
            normalize_vectors=False, eigensolver="arpack",
        )
        wks = gm.wks([ev, evec], evals=EVALS, variance=VAR, l2=True)
        wks = np.nan_to_num(wks)
        vecs.append(gm.knn_histogram(wks, codebook, knn=BOF_KNN))
    return np.asarray(vecs)

def encode_molsg(surf_dir, fns):
    """MolSG FEM-LBO WKS -> hard-assignment BoF baseline."""
    import sys
    sys.path.insert(0, str(Path(__file__).resolve().parent if "__file__" in globals() else "."))
    from sklearn.preprocessing import normalize as sk_normalize
    from molsg.localgeometry import WKS_descriptor
    from molsg.laplacemesh import compute_lb_fem
    import molsg.bagoffeatures as bf

    codebook = np.load(MOLSG_CODEBOOK, allow_pickle=True)
    vecs = []
    for f in fns:
        mol = np.load(str(surf_dir / f), allow_pickle=True)
        eigs = compute_lb_fem(vertices=mol[0], faces=mol[1], k=K)
        wks = sk_normalize(WKS_descriptor(eigs, evals=EVALS, variance=VAR, ))
        vecs.append(bf.hq_descriptor(wks, codebook).flatten())
    return np.asarray(vecs)

ENCODERS = {"GE-MolSG": encode_gemolsg, "MolSG": encode_molsg}

## 5 · Per-target driver (iterative, cached)

In [ ]:
def run_target(target):
    out_by_method = {}
    surf_dir = fns = query_ids = None
    for method in METHODS:
        csv_path = RESULTS_ROOT / method / f"{target}.csv"
        if csv_path.exists() and not FORCE:
            log.info("[%s | %s] cached -> %s", target, method, csv_path)
            out_by_method[method] = pd.read_csv(csv_path)
            continue
        if surf_dir is None:
            surf_dir = resolve_target_dir(target, DATA_ROOT, OUT_DIR / "_work")
            fns = list_surface_fns(surf_dir)
            query_ids = load_query_ids(resolve_queries_csv(QUERIES_DIR, target))
            n_lig = sum(f[0] == "l" for f in fns)
            log.info("[%s] %d surfaces (%d ligands, %d decoys), %d queries",
                     target, len(fns), n_lig, len(fns) - n_lig, len(query_ids))
        log.info(fns)
        log.info("[%s | %s] encoding %d surfaces ...", target, method, len(fns))
        vectors = ENCODERS[method](surf_dir, fns)
        log.info("[%s | %s] retrieval (%s) ...", target, method, SIM_KIND[method])
        df = retrieve(vectors, fns, query_ids, SIM_KIND[method])
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv_path, index=False)
        log.info("[%s | %s] %d queries  mean EF1%%=%.3f  mean BEDROC=%.3f -> %s",
                 target, method, len(df),
                 df["EF1%"].mean() if len(df) else float("nan"),
                 df["BEDROC"].mean() if len(df) else float("nan"), csv_path)
        out_by_method[method] = df
    return out_by_method

sampled_data = {m: {} for m in METHODS}
for ti, target in enumerate(TARGETS, 1):
    log.info("==== target %d/%d : %s ====", ti, len(TARGETS), target)
    try:
        by_method = run_target(target)
    except FileNotFoundError as e:
        log.error("skipping %s: %s", target, e)
        continue
    for method, df in by_method.items():
        if len(df):
            sampled_data[method][target] = df

log.info("Encoding + retrieval complete")

## 6 · Summary

In [ ]:
rows = []
for method in sampled_data:
    for t in TARGETS:
        if t in sampled_data[method] and len(sampled_data[method][t]):
            df = sampled_data[method][t]
            rows.append(dict(method=method, target=t,
                             mean_EF1=df["EF1%"].mean(),
                             mean_BEDROC=df["BEDROC"].mean(),
                             n_queries=len(df)))
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary_per_target.csv", index=False)
log.info("wrote summary_per_target.csv (%d rows)", len(summary))
summary

## 7 · Scatter / line plots — BEDROC and EF1%
Per-target mean over the target's queries, for GE-MolSG and MolSG.

In [ ]:
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", font_scale=1.0)
except Exception:
    pass

COLOUR_MAP = {"GE-MolSG": "#0072B2", "MolSG": "#9467BD"}
METHOD_MARKERS = {"GE-MolSG": "s", "MolSG": "p"}
SCATTER_METHODS = ["GE-MolSG", "MolSG"]

def scatter(metric):
    tsorted = sorted(TARGETS)
    x_pos = np.arange(len(tsorted))
    y_min = 0.0
    y_max = 50.0 if metric == "EF1%" else 0.9

    fig_width = max(8, len(tsorted) * 0.35)
    fig, ax = plt.subplots(figsize=(fig_width, 4.5), facecolor="white")
    ax.set_facecolor("#EBEBEB")
    ax.grid(axis="y", linestyle="-", color="white", linewidth=0.8, zorder=0)
    ax.grid(axis="x", linestyle="-", color="white", linewidth=0.8, zorder=0)
    ax.margins(x=0.02)

    for method in SCATTER_METHODS:
        means = [sampled_data[method][t][metric].mean()
                 if t in sampled_data.get(method, {}) and len(sampled_data[method][t])
                 else np.nan
                 for t in tsorted]
        col = COLOUR_MAP.get(method, "#888888")
        ax.plot(x_pos, means, color=col, linewidth=1.4,
                marker=METHOD_MARKERS.get(method, "o"), markersize=6,
                markerfacecolor="white", markeredgecolor=col, markeredgewidth=1.5,
                label=method, zorder=3)

    ax.set_xlim(-0.5, len(tsorted) - 0.5)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(tsorted, rotation=90, ha="center", fontsize=9)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xlabel("Target", fontsize=12)
    title = {"BEDROC": "DUDE-Z Mean BEDROC (α=20) Retrieval Performance",
             "EF1%": "DUDE-Z Mean EF1% Retrieval Performance"}.get(
                 metric, f"DUDE-Z Mean {metric} Retrieval Performance")
    ax.set_title(title, fontsize=14, pad=6)
    ax.legend(fontsize=10, bbox_to_anchor=(1.01, 1), loc="upper left",
              borderaxespad=0.0, framealpha=0.85)
    plt.tight_layout()
    path = OUT_DIR / f"scatter_{metric.replace('%','pct')}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    log.info("saved %s", path)

scatter("BEDROC")
scatter("EF1%")

## 8 · Wilcoxon signed-rank test — focused on GE-MolSG
Paired on per-target means; GE-MolSG vs each other method, per metric. p-values unadjusted.

In [ ]:
from scipy.stats import wilcoxon

WILCOXON_METHOD = "GE-MolSG"
alpha = 0.05
records = []

for metric in METRICS:
    for baseline in sampled_data:
        if baseline == WILCOXON_METHOD:
            continue
        paired_focal, paired_base = [], []
        for target in TARGETS:
            if (target in sampled_data[WILCOXON_METHOD] and len(sampled_data[WILCOXON_METHOD].get(target, [])) and
                    target in sampled_data[baseline] and len(sampled_data[baseline].get(target, []))):
                paired_focal.append(sampled_data[WILCOXON_METHOD][target][metric].mean())
                paired_base.append(sampled_data[baseline][target][metric].mean())
        n_pairs = len(paired_focal)
        if n_pairs < 4:
            records.append(dict(Metric=metric, Baseline=baseline, N_targets=n_pairs,
                                W=np.nan, p_value=np.nan, Significant="—",
                                Direction="insufficient data"))
            continue
        diffs = np.array(paired_focal) - np.array(paired_base)
        if np.all(diffs == 0):
            records.append(dict(Metric=metric, Baseline=baseline, N_targets=n_pairs,
                                W=np.nan, p_value=np.nan, Significant="—",
                                Direction="identical"))
            continue
        stat, p = wilcoxon(paired_focal, paired_base, alternative="two-sided")
        records.append(dict(Metric=metric, Baseline=baseline, N_targets=n_pairs,
                            W=round(stat, 3), p_value=round(p, 4),
                            Significant="✓" if p < alpha else "✗",
                            Direction="better" if np.mean(diffs) > 0 else "worse"))

sig_df = pd.DataFrame(records)
if len(sig_df):
    sig_df = sig_df.sort_values(["Metric", "p_value"])
sig_df.to_csv(OUT_DIR / f"wilcoxon_{WILCOXON_METHOD.replace(' ','_')}.csv", index=False)

print(f"Wilcoxon signed-rank test: {WILCOXON_METHOD} vs all others")
print(f"Paired on per-target means; α = {alpha}\n")
for metric in METRICS:
    if "Metric" not in sig_df.columns:
        break
    sub = sig_df[sig_df.Metric == metric][["Baseline", "N_targets", "W", "p_value", "Significant", "Direction"]]
    print(f"── {metric} {'─'*40}")
    print(sub.to_string(index=False))
    print()
sig_df